In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [2]:
column_names = [
    "CrimeRate",  # The per capita crime rate by town. lower housing values
    "ResidentialLandZoned", # The proportion of residential land zoned for lots over 25,000 sq.ft.
    "IndustrialLandUse", # The proportion of non-retail business acres per town.
    "CharlesRiver", # Charles River dummy variable (1 if tract bounds river; 0 otherwise)
    "NitrixOxide", # Nitric oxides concentration (parts per 10 million)
    "AvgRoomPerDwelling", # The average number of rooms per dwelling
    "HousingAge", # The proportion of owner-occupied units built prior to 1940
    "DistanceToWork", # Weighted distances to five Boston employment centres
    "HighwayAccess", # Index of accessibility to radial highways
    "PropertyTaxRate", # Full-value property-tax rate per $10,000
    "Pupil-TeacherRatio", # The pupil-teacher ratio by town
    "B", # 1000(Bk - 0.63)^2 where Bk is the proportion of blacks by town
    "LowSocioEcomic", # The percentage of lower status of the population
    "Value", # Median value of owner-occupied homes in $1000s
]
df = pd.read_csv("data/housing.csv", sep=r"\s+", header=None, names=column_names)

In [3]:
df.head(2)

,CrimeRate,ResidentialLandZoned,IndustrialLandUse,CharlesRiver,NitrixOxide,AvgRoomPerDwelling,HousingAge,DistanceToWork,HighwayAccess,PropertyTaxRate,Pupil-TeacherRatio,B,LowSocioEcomic,Value
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.9,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.9,9.14,21.6


In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import mlflow

In [4]:
mlflow.set_tracking_uri("http://localhost:5020")
mlflow.set_experiment("Boston Model Experiment")
mlflow.set_registry_uri("sqlite:///mlflow_registry.db")


In [ ]:
feature_cols = ["CrimeRate", "ResidentialLandZoned", "IndustrialLandUse", "CharlesRiver", "NitrixOxide", "AvgRoomPerDwelling", "HousingAge", "DistanceToWork", "HighwayAccess", "PropertyTaxRate", "Pupil-TeacherRatio", "B", "LowSocioEcomic"]
X = df[feature_cols]
y = df['Value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
mlflow.start_run(run_name="XGBRegressor Run")

xgb_model = XGBRegressor(n_estimators=150, max_depth=5, learning_rate=0.05, random_state=42)
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5

mlflow.log_metric("rmse", rmse)
mlflow.xgboost.log_model(xgb_model, "xgbr_model")

model_name = "XGBRegressor Model"
model_uri = f"runs:/{mlflow.active_run().info.run_id}/xgbr_model"
mlflow.register_model(model_uri, model_name)

mlflow.end_run()



2025/09/03 09:43:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/09/03 09:43:46 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'XGBRegressor Model'.
2025/09/03 09:43:46 WARNING mlflow.tracking._model_registry.fluent: Run with id d2de8e82dfcf4bb1ac6dddcc5f3a74d9 has no artifacts at artifact path 'xgbr_model', registering model based on models:/m-384492735ce94636a12e7a5c15779a30 instead
2025/09/03 09:43:46 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBRegressor Model, version 1
Created version '1' of model 'XGBRegressor Model'.


🏃 View run XGBRegressor Run at: http://localhost:5020/#/experiments/417350048653659120/runs/d2de8e82dfcf4bb1ac6dddcc5f3a74d9
🧪 View experiment at: http://localhost:5020/#/experiments/417350048653659120


In [ ]:
# Load registered model and make predictions
loaded_model = mlflow.pyfunc.load_model(model_uri)
loaded_model.predict(X_test)